# EMG Practical Workshop: Signal Processing and Feature Extraction

This notebook is designed for 3rd-year Bachelor IT students to practice core EMG data analysis workflows.

You will learn how to:
- inspect and visualize EMG signals (EMG1-EMG3),
- apply and compare normalization methods,
- extract useful signal features,
- run a simple baseline classifier for movement recognition.

## Learning Outcomes
After finishing this practical, students should be able to:
1. Explain the structure of this EMG dataset.
2. Visualize raw EMG signals and identify differences between movements.
3. Explain why normalization is needed and compare common normalization methods.
4. Extract time-domain and frequency-domain EMG features.
5. Evaluate a simple machine learning baseline for movement classification.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)

DATASET_DIR = Path('dataset_experiment')
assert DATASET_DIR.exists(), f'Dataset folder not found: {DATASET_DIR.resolve()}'

print('Dataset directory:', DATASET_DIR.resolve())

In [ ]:
pattern = re.compile(r'^(subject\d+)_(.+)_(\d+)\.csv$')
records = []

for csv_path in DATASET_DIR.rglob('*.csv'):
    match = pattern.match(csv_path.name)
    if not match:
        continue
    subject, movement, trial = match.groups()
    records.append({
        'subject': subject,
        'movement': movement,
        'trial': int(trial),
        'file_path': csv_path
    })

index_df = pd.DataFrame(records).sort_values(['subject', 'movement', 'trial']).reset_index(drop=True)

print('Total files:', len(index_df))
display(index_df.head(10))
display(index_df.groupby(['subject', 'movement']).size().unstack(fill_value=0))

## Step 1 - Load EMG CSV with Metadata
Each CSV file starts with metadata lines (starting with `#`), followed by a normal table with columns: `sample_idx, EMG1, EMG2, EMG3`.

We parse both parts so we can use sampling-rate metadata in later analysis.

In [ ]:
def parse_metadata_value(raw: str):
    # Metadata values can look like ',998.862,,' so we pick the first non-empty token.
    tokens = [t.strip() for t in raw.split(',') if t.strip()]
    if not tokens:
        return ''
    candidate = tokens[0]
    try:
        return float(candidate)
    except ValueError:
        return candidate


def load_emg_file(file_path: Path):
    metadata = {}
    comment_lines = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.startswith('#'):
                break
            comment_lines += 1
            line_clean = line[1:].strip()
            if ':' in line_clean:
                key, value = line_clean.split(':', 1)
                metadata[key.strip()] = parse_metadata_value(value)

    df = pd.read_csv(file_path, skiprows=comment_lines)

    fs = metadata.get('Actual Sampling Rate (Hz)', metadata.get('Requested Sampling Rate (Hz)', 1000.0))
    fs = float(fs) if isinstance(fs, (int, float, str)) else 1000.0

    if 'sample_idx' in df.columns:
        df['time_s'] = df['sample_idx'] / fs
    else:
        df['time_s'] = np.arange(len(df)) / fs

    return df, metadata, fs


sample_row = index_df.iloc[0]
sample_df, sample_meta, sample_fs = load_emg_file(sample_row['file_path'])

print('Sample file:', sample_row['file_path'])
print('Sampling rate:', sample_fs, 'Hz')
print('Metadata keys:', list(sample_meta.keys()))
display(sample_df.head())

## Step 2 - Visualize Raw EMG Signals (EMG1-EMG3)

In [ ]:
def plot_emg_channels(df: pd.DataFrame, title: str, channels=('EMG1', 'EMG2', 'EMG3')):
    fig, axes = plt.subplots(len(channels), 1, figsize=(14, 7), sharex=True)
    if len(channels) == 1:
        axes = [axes]

    for ax, ch in zip(axes, channels):
        ax.plot(df['time_s'], df[ch], linewidth=0.9)
        ax.set_ylabel(ch)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


subject_to_plot = 'subject01'
movement_to_plot = '2-finger-flexion'
trial_to_plot = 1

row = index_df.query('subject == @subject_to_plot and movement == @movement_to_plot and trial == @trial_to_plot').iloc[0]
df_raw, meta_raw, fs_raw = load_emg_file(row['file_path'])
plot_emg_channels(df_raw, f'Raw EMG | {subject_to_plot} | {movement_to_plot} | trial {trial_to_plot}')

### How to Interpret This Raw EMG Plot
Purpose:
- To observe the original (unprocessed) electrical activity from three channels (EMG1-EMG3).
- To identify burst regions (muscle activation), baseline periods (rest), and possible artifacts/noise.

Meaning:
- High-amplitude bursts usually indicate stronger or more abrupt muscle activation.
- Differences across EMG1, EMG2, and EMG3 suggest that each channel captures activity from slightly different muscle regions.
- Before extracting features, this plot helps verify data quality and timing structure of the movement trial.

In [ ]:
def moving_average(x: np.ndarray, window: int = 100):
    window = max(1, int(window))
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode='same')


subject_view = 'subject01'
trial_view = 1
movements = sorted(index_df['movement'].unique())

fig, ax = plt.subplots(figsize=(14, 6))
for movement in movements:
    q = index_df.query('subject == @subject_view and movement == @movement and trial == @trial_view')
    if q.empty:
        continue
    df_temp, _, _ = load_emg_file(q.iloc[0]['file_path'])
    envelope = moving_average(np.abs(df_temp['EMG1'].to_numpy()), window=120)
    ax.plot(df_temp['time_s'], envelope, label=movement, linewidth=1.4)

ax.set_title(f'Envelope Comparison (EMG1) | {subject_view} trial {trial_view}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Rectified + Smoothed Amplitude')
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

### How to Interpret Envelope Comparison (EMG1)
Purpose:
- To compare movement patterns using a smoothed envelope of rectified EMG1 (absolute signal + moving average).
- To make activation trend comparisons easier than raw high-frequency waveforms.

Meaning:
- A higher envelope means stronger average muscle activation over short time windows.
- Timing differences (when peaks happen) show when each gesture activates most strongly.
- This view is useful for gesture separability: if envelopes are consistently different, classification becomes easier.

## Step 3 - Normalization Methods
Why normalize EMG?
- Different sensors, skin conditions, and placements can change amplitude scale.
- Without normalization, features may reflect scale artifacts instead of movement differences.

In this dataset, files already include a calibration/normalization process in metadata, but we still compare common methods for learning:
- **Min-Max**: bounded range [0, 1], simple, but sensitive to outliers.
- **Z-score**: centers and scales by standard deviation, useful for many ML models.
- **Robust (Median/IQR)**: less sensitive to outliers, more stable for noisy signals.

In [ ]:
def normalize_minmax(x: np.ndarray):
    x_min, x_max = np.min(x), np.max(x)
    eps = 1e-12
    return (x - x_min) / (x_max - x_min + eps)


def normalize_zscore(x: np.ndarray):
    mu, sigma = np.mean(x), np.std(x)
    eps = 1e-12
    return (x - mu) / (sigma + eps)


def normalize_robust(x: np.ndarray):
    med = np.median(x)
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    eps = 1e-12
    return (x - med) / (iqr + eps)


x = df_raw['EMG1'].to_numpy()
x_minmax = normalize_minmax(x)
x_z = normalize_zscore(x)
x_rb = normalize_robust(x)

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
axes[0].plot(df_raw['time_s'], x, color='black', linewidth=1)
axes[0].set_title('Original EMG1 signal')
axes[1].plot(df_raw['time_s'], x_minmax, linewidth=1)
axes[1].set_title('Min-Max normalized')
axes[2].plot(df_raw['time_s'], x_z, linewidth=1)
axes[2].set_title('Z-score normalized')
axes[3].plot(df_raw['time_s'], x_rb, linewidth=1)
axes[3].set_title('Robust normalized (median/IQR)')
axes[3].set_xlabel('Time (s)')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Normalization Methods: What They Do and Why
1. Min-Max normalization
- Formula: x' = (x - min(x)) / (max(x) - min(x)).
- Output range is approximately [0, 1].
- Goal: put all values on the same bounded scale for easier comparison and plotting.
- Limitation: highly sensitive to outliers because min/max can be extreme.

2. Z-score normalization
- Formula: x' = (x - mean(x)) / std(x).
- Output is centered around 0 with unit variance.
- Goal: emphasize how far a value is from the average in standard-deviation units.
- Useful when models assume standardized input.
- Limitation: can still be influenced by strong outliers.

3. Robust normalization (median/IQR)
- Formula: x' = (x - median(x)) / IQR, where IQR = Q3 - Q1.
- Goal: scale data using robust statistics that are less affected by outliers.
- Useful for noisy biosignals when spikes/artifacts exist.
- Limitation: interpretation is less intuitive for beginners than Min-Max.

Practical objective in EMG:
- Reduce amplitude-scale bias across sessions/subjects/sensors,
- preserve movement-related patterns,
- improve fairness and stability of downstream feature extraction and classification.

## Step 4 - Feature Extraction
We will extract two groups of features:
- **Time-domain** (fast and widely used): MAV, RMS, Variance, Waveform Length, IEMG, ZC, SSC.
- **Frequency-domain** (captures spectral behavior): Mean Frequency, Median Frequency, Total Power.

General trade-off:
- Time-domain features are simpler and efficient.
- Frequency-domain features can capture richer muscle activation patterns, but may need more computation and careful windowing.

In [ ]:
def feat_mav(x):
    return np.mean(np.abs(x))


def feat_rms(x):
    return np.sqrt(np.mean(x ** 2))


def feat_var(x):
    return np.var(x)


def feat_wl(x):
    return np.sum(np.abs(np.diff(x)))


def feat_iemg(x):
    return np.sum(np.abs(x))


def feat_zc(x, threshold=1e-3):
    x1 = x[:-1]
    x2 = x[1:]
    crossings = ((x1 * x2) < 0) & (np.abs(x1 - x2) >= threshold)
    return np.sum(crossings)


def feat_ssc(x, threshold=1e-3):
    x_prev = x[:-2]
    x_curr = x[1:-1]
    x_next = x[2:]
    cond = (((x_curr - x_prev) * (x_curr - x_next)) > 0) & (
        (np.abs(x_curr - x_prev) >= threshold) | (np.abs(x_curr - x_next) >= threshold)
    )
    return np.sum(cond)


def spectral_features(x, fs):
    x = np.asarray(x)
    x = x - np.mean(x)

    fft_vals = np.fft.rfft(x)
    power = np.abs(fft_vals) ** 2
    freqs = np.fft.rfftfreq(len(x), d=1.0 / fs)

    total_power = np.sum(power) + 1e-12
    mean_freq = np.sum(freqs * power) / total_power

    cumulative = np.cumsum(power)
    median_freq = freqs[np.searchsorted(cumulative, cumulative[-1] / 2)]

    return mean_freq, median_freq, total_power


def extract_features_for_channel(x, fs):
    mf, mdf, pwr = spectral_features(x, fs)
    return {
        'MAV': feat_mav(x),
        'RMS': feat_rms(x),
        'VAR': feat_var(x),
        'WL': feat_wl(x),
        'IEMG': feat_iemg(x),
        'ZC': feat_zc(x),
        'SSC': feat_ssc(x),
        'MNF': mf,
        'MDF': mdf,
        'PWR': pwr
    }

In [ ]:
feature_rows = []

for _, r in index_df.iterrows():
    df_sig, _, fs = load_emg_file(r['file_path'])
    row_features = {
        'subject': r['subject'],
        'movement': r['movement'],
        'trial': r['trial']
    }

    for ch in ['EMG1', 'EMG2', 'EMG3']:
        ch_features = extract_features_for_channel(df_sig[ch].to_numpy(), fs)
        for k, v in ch_features.items():
            row_features[f'{ch}_{k}'] = v

    feature_rows.append(row_features)

features_df = pd.DataFrame(feature_rows)
print('Feature table shape:', features_df.shape)
display(features_df.head())

In [ ]:
# Feature visualization for one trial: time-domain and frequency-domain
example = features_df.iloc[0]
channels = ['EMG1', 'EMG2', 'EMG3']

# Time-domain feature map
td_features = ['MAV', 'RMS', 'VAR', 'WL', 'IEMG', 'ZC', 'SSC']
td_matrix = np.array([[example[f'{ch}_{feat}'] for feat in td_features] for ch in channels], dtype=float)

plt.figure(figsize=(11, 3.8))
plt.imshow(td_matrix, aspect='auto', cmap='YlGnBu')
plt.colorbar(label='Feature value')
plt.xticks(range(len(td_features)), td_features, rotation=25, ha='right')
plt.yticks(range(len(channels)), channels)
plt.title(f'Time-domain feature map | {example["subject"]} | {example["movement"]} | trial {int(example["trial"])}')
plt.tight_layout()
plt.show()

# Frequency-domain feature map
fd_features = ['MNF', 'MDF', 'PWR']
fd_matrix = np.array([[example[f'{ch}_{feat}'] for feat in fd_features] for ch in channels], dtype=float)

plt.figure(figsize=(7.2, 3.8))
plt.imshow(fd_matrix, aspect='auto', cmap='OrRd')
plt.colorbar(label='Feature value')
plt.xticks(range(len(fd_features)), fd_features)
plt.yticks(range(len(channels)), channels)
plt.title(f'Frequency-domain feature map | {example["subject"]} | {example["movement"]} | trial {int(example["trial"])}')
plt.tight_layout()
plt.show()

### How to Interpret These Feature Maps
Purpose:
- To visualize extracted numeric features so students can connect equations to observable patterns.

Meaning:
- Rows are channels (EMG1-EMG3), columns are features.
- Brighter cells indicate larger feature values.
- If a movement produces a distinct pattern across many cells, that movement is easier to separate in classification.

Tip:
- You can compare this map for two different gestures and identify which features change most.

In [ ]:
feature_to_plot = 'EMG1_RMS'
plot_data = [
    features_df.loc[features_df['movement'] == m, feature_to_plot].values
    for m in sorted(features_df['movement'].unique())
]
labels = sorted(features_df['movement'].unique())

plt.figure(figsize=(14, 6))
plt.boxplot(plot_data, tick_labels=labels, showfliers=False)
plt.xticks(rotation=25, ha='right')
plt.ylabel(feature_to_plot)
plt.title(f'Movement-wise distribution of {feature_to_plot}')
plt.tight_layout()
plt.show()

### Meaning of Movement-wise Distribution of EMG1_RMS
What this plot shows:
- A boxplot of EMG1_RMS values grouped by movement class.

How to read it:
- Center line in each box: median RMS for that movement.
- Box height: middle 50% range (IQR).
- Whiskers: broader spread of non-outlier values.

Interpretation objective:
- If two gestures have clearly different medians/ranges, EMG1_RMS helps discriminate them.
- If distributions overlap heavily, this single feature is weak alone and should be combined with other features/channels.

## Step 5 - Baseline Movement Classification (Optional but Recommended)
A simple classifier helps students see whether extracted features carry useful movement information.

We use a train-test split and Random Forest as a baseline model.

In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

    X = features_df.drop(columns=['subject', 'movement', 'trial'])
    y = features_df['movement']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    print('Train samples:', len(X_train), '| Test samples:', len(X_test))
    print('Class counts in test split:')
    print(y_test.value_counts().sort_index())

    clf = RandomForestClassifier(n_estimators=300, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'Baseline accuracy: {acc:.4f}')
    print('\nClassification report:')
    print(classification_report(y_test, y_pred))

    fig, ax = plt.subplots(figsize=(8, 8))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, xticks_rotation=30)
    plt.title('Confusion Matrix - Random Forest Baseline')
    plt.tight_layout()
    plt.show()

except ModuleNotFoundError as e:
    print('scikit-learn is not installed in this environment.')
    print('Install with: pip install scikit-learn')
    print('Original error:', e)

### Classification Validation and Confusion Matrix Interpretation
Validation used in this notebook:
- Stratified train-test split with test_size = 0.25.
- Stratified means class proportions are preserved between train and test.

Why can confusion-matrix values reach 4 (for example in ulnar-deviation)?
- Dataset size per gesture = 3 subjects x 5 trials = 15 samples.
- Test split is 25%, so about 15 x 0.25 = 3.75, rounded to 4 test samples for many classes.
- Therefore each row in the confusion matrix can have totals around 4, not 3.
- The number is based on test samples, not number of subjects.

Important note:
- This validation is sample-level random split, not strict subject-independent validation.
- For stronger generalization testing, use leave-one-subject-out or train on subject01+02 and test on subject03.

## Step 6 - Generalization Testing Across Subjects
This section evaluates whether the model can generalize to unseen people, not only unseen trials.

We add two stronger validation settings:
1. Subject-independent split (train on selected subjects, test on one unseen subject).
2. Leave-One-Subject-Out (LOSO) cross-validation.

Why this matters:
- Random sample split can place samples from the same subject in both train and test sets.
- LOSO is stricter and better reflects real deployment where a new user was not in training.

In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
    from sklearn.model_selection import LeaveOneGroupOut

    X_all = features_df.drop(columns=['subject', 'movement', 'trial'])
    y_all = features_df['movement']
    groups = features_df['subject']
    class_labels = sorted(y_all.unique())

    print('=== A) Subject-Independent Split (Hold-out Subject) ===')
    holdout_subject = 'subject03'
    if holdout_subject not in groups.unique():
        holdout_subject = sorted(groups.unique())[-1]

    train_mask = groups != holdout_subject
    test_mask = groups == holdout_subject

    X_train_subj, y_train_subj = X_all[train_mask], y_all[train_mask]
    X_test_subj, y_test_subj = X_all[test_mask], y_all[test_mask]

    clf_subj = RandomForestClassifier(n_estimators=300, random_state=42)
    clf_subj.fit(X_train_subj, y_train_subj)
    y_pred_subj = clf_subj.predict(X_test_subj)

    acc_subj = accuracy_score(y_test_subj, y_pred_subj)
    print(f'Train subjects: {sorted(groups[train_mask].unique())}')
    print(f'Test subject: {holdout_subject}')
    print(f'Subject-independent accuracy: {acc_subj:.4f}')
    print('Class counts in hold-out subject test set:')
    print(y_test_subj.value_counts().sort_index())
    print('\nClassification report (hold-out subject):')
    print(classification_report(y_test_subj, y_pred_subj, zero_division=0))

    fig, ax = plt.subplots(figsize=(8, 8))
    cm_subj = confusion_matrix(y_test_subj, y_pred_subj, labels=class_labels)
    ConfusionMatrixDisplay(confusion_matrix=cm_subj, display_labels=class_labels).plot(ax=ax, xticks_rotation=30)
    plt.title(f'Confusion Matrix - Subject-Independent ({holdout_subject} as test)')
    plt.tight_layout()
    plt.show()

    print('\n=== B) Leave-One-Subject-Out (LOSO) Cross-Validation ===')
    logo = LeaveOneGroupOut()
    loso_rows = []
    cm_loso_sum = np.zeros((len(class_labels), len(class_labels)), dtype=int)

    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups=groups), start=1):
        X_train_fold = X_all.iloc[train_idx]
        X_test_fold = X_all.iloc[test_idx]
        y_train_fold = y_all.iloc[train_idx]
        y_test_fold = y_all.iloc[test_idx]

        test_subject_name = groups.iloc[test_idx].iloc[0]

        clf_fold = RandomForestClassifier(n_estimators=300, random_state=42)
        clf_fold.fit(X_train_fold, y_train_fold)
        y_pred_fold = clf_fold.predict(X_test_fold)

        fold_acc = accuracy_score(y_test_fold, y_pred_fold)
        loso_rows.append({
            'fold': fold_idx,
            'test_subject': test_subject_name,
            'n_test_samples': len(test_idx),
            'accuracy': fold_acc
        })

        cm_fold = confusion_matrix(y_test_fold, y_pred_fold, labels=class_labels)
        cm_loso_sum += cm_fold

    loso_df = pd.DataFrame(loso_rows)
    display(loso_df)
    print(f'LOSO mean accuracy: {loso_df["accuracy"].mean():.4f}')
    print(f'LOSO std accuracy:  {loso_df["accuracy"].std(ddof=1):.4f}')

    plt.figure(figsize=(7, 4))
    plt.bar(loso_df['test_subject'], loso_df['accuracy'])
    plt.ylim(0, 1)
    plt.ylabel('Accuracy')
    plt.title('LOSO accuracy per held-out subject')
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm_loso_sum, display_labels=class_labels).plot(ax=ax, xticks_rotation=30)
    plt.title('Aggregated Confusion Matrix - LOSO (all folds combined)')
    plt.tight_layout()
    plt.show()

except ModuleNotFoundError as e:
    print('Required package is not installed in this environment.')
    print('Install with: pip install scikit-learn')
    print('Original error:', e)

### How to Interpret Subject-Generalization Results
1. Subject-independent split result:
- This tests transfer from known subjects to one unseen subject.
- If accuracy drops versus random split, the model learned subject-specific patterns.

2. LOSO result:
- Each fold uses one subject as test and all others as train.
- Mean LOSO accuracy is a stronger estimate of real-world generalization.
- Standard deviation shows consistency across different held-out subjects.

3. Aggregated LOSO confusion matrix:
- Sums errors from all held-out-subject folds.
- Helps identify gestures that are consistently confused across users.

## Discussion: Feature Technique Strengths and Weaknesses
### Time-domain features
- Pros: simple, fast, easy to interpret, commonly used in real-time EMG systems.
- Cons: may miss detailed spectral changes and can be sensitive to noise if not preprocessed properly.

### Frequency-domain features
- Pros: capture muscle activation frequency behavior and fatigue-related trends.
- Cons: require transforms (FFT), windowing decisions, and can be less intuitive for beginners.

### Practical recommendation
For 3rd-year students, start with time-domain features, then add selected frequency-domain features for improved representation.
1. Compare accuracy using only time-domain features vs only frequency-domain features.
2. Replace Random Forest with Logistic Regression or SVM and compare results.
3. Test subject-independent split (train on some subjects, test on unseen subjects).
4. Add signal filtering (for example, a band-pass filter) before feature extraction and observe impact.
5. Write a short conclusion: which normalization and feature set worked best, and why?